<a href="https://colab.research.google.com/github/nguyenlmhcm/MBHT-KDD22/blob/bt-mbht/notebooks/Gate3_Tmall_phase1_multiseed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gate 3 — Tmall pha 1: A0 vs A1 multi-seed

## Chạy ĐÚNG protocol gốc — không cắt epoch

Early-stopping **BẬT**, `stopping_step=10` đúng như MBHT gốc. `SAFETY_CAP=15` chỉ là lưới chặn
trường hợp xấu, không phải tham số thí nghiệm. Nếu early-stopping tự kích hoạt trước cap thì ta
chạy nguyên protocol gốc và **so được với số published**.

### Vì sao bỏ epoch cap: "epoch" là đơn vị sai để so giữa hai dataset

| | batch/epoch | epoch đã chạy | **tổng bước gradient** |
|---|---:|---:|---:|
| RetailRocket | 480 | 26 | **12,480** |
| Tmall | 7,234 | ? | ? |

**12,480 ÷ 7,234 = 1.7 epoch Tmall.** Toàn bộ quá trình huấn luyện RetailRocket tương đương chưa
tới 2 epoch Tmall. Hội tụ bị chi phối bởi **số bước cập nhật**, không phải số vòng quét dữ liệu —
nên nhiều khả năng Tmall hội tụ trong ít epoch, và không cần cắt gì cả.

Ước tính "47–77 giờ/run" trước đây dựa trên giả định "25 epoch như retail". Giả định đó nhiều khả
năng sai theo hướng có lợi. Notebook này **đo thay vì đoán**.

---

## ⚠ KHOÁ TRƯỚC KHI CHẠY — bất đối xứng khi đọc kết quả

Ghi ở đây và lưu ra `run_meta/` **trước** khi có bất kỳ con số nào, để không thể diễn giải lại
sau khi thấy kết quả:

> Nếu về sau buộc phải cắt epoch vì compute, thì kết quả phải đọc theo **hai chiều khác nhau**:
>
> - **ÂM tính dưới cap = bằng chứng YẾU.** B1 có thêm tham số, bị cắt ngắn thì thiệt hơn A0.
>   Không được kết luận "B1 vô dụng" từ một kết quả âm dưới cap.
> - **DƯƠNG tính dưới cap = bằng chứng MẠNH.** Thắng dù bị trói tay.
>
> Ở protocol đầy đủ (early-stopping tự dừng), bất đối xứng này không áp dụng.

---

## Tiêu chí đã khoá (đọc từ `experiments/seeds.json`, commit trước mọi lần chạy)

- **Seed** `[2020, 2021, 2022]`, dùng chung cả A0 lẫn A1. Không đổi sau khi thấy kết quả.
- **Metric chính** `ndcg@10`.
- **Báo cáo** mean ± std + **Welch's t-test** (hai mẫu độc lập, phương sai không đều).
- **GO khi VÀ CHỈ KHI**: hiệu mean vượt **sàn nhiễu đo trên chính Tmall ở pha này** **VÀ** p < 0.05.

σ của RetailRocket **không** dùng làm sàn nhiễu Tmall — chỉ dùng để ước lượng số seed.

---

## Cách chạy

Run đầu tiên (**A0 seed 2020**) vừa là **probe hội tụ** vừa là **run pha 1 thật** — không phí gì.
Chạy tới hết phần probe rồi **DỪNG**, gửi đường validation về trước khi chạy 5 run còn lại.

## 1. GPU + Drive

In [1]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

Tue Aug 18 06:24:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Đường dẫn + clone code

In [2]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/DeAnThS'
DATA_DIR     = os.path.join(PROJECT_ROOT, 'dataset')
CKPT_DIR     = os.path.join(PROJECT_ROOT, 'checkpoints')
LOG_DIR      = os.path.join(PROJECT_ROOT, 'logs')
RUN_META_DIR = os.path.join(PROJECT_ROOT, 'run_meta')
CODE_DIR     = '/content/MBHT-KDD22'
for d in [DATA_DIR, CKPT_DIR, LOG_DIR, RUN_META_DIR]:
    os.makedirs(d, exist_ok=True)

DATASET_ROOT = os.path.join(DATA_DIR, 'MBHT_dataset')
assert os.path.isdir(DATASET_ROOT), f'Chua co {DATASET_ROOT} -- chay notebook Gate 1-2 truoc.'

REPO_URL      = 'https://github.com/nguyenlmhcm/MBHT-KDD22.git'
BRANCH        = 'bt-mbht'
PINNED_COMMIT = '3dfdb67edc1ae7bcbe7e5a3a7975f82ef11819ff'
DATASET_NAME  = 'tmall_beh'

# Protocol goc MBHT -- KHONG cat epoch.
STOPPING_STEP = 10   # patience, dung nhu MBHT goc
SAFETY_CAP    = 15   # luoi chan truong hop xau, khong phai tham so thi nghiem

# Moc doi chieu tu RetailRocket: 480 batch/epoch x 26 epoch
RETAIL_STEPS = 12480

print('dataset:', DATASET_NAME, '| patience:', STOPPING_STEP, '| safety cap:', SAFETY_CAP)

dataset: tmall_beh | patience: 10 | safety cap: 15


In [3]:
import subprocess, time

# Cell cai dat ben duoi co `%cd {CODE_DIR}`; buoc ra truoc de chay lai cell nay luon an toan.
os.chdir('/content')
if os.path.isdir(CODE_DIR):
    subprocess.run(['rm', '-rf', CODE_DIR], check=True)

for attempt in range(1, 6):
    r = subprocess.run(['git','clone','-b',BRANCH,REPO_URL,CODE_DIR], capture_output=True, text=True)
    if r.returncode == 0:
        print(f'Clone OK (attempt {attempt})'); break
    print(f'attempt {attempt} failed:', r.stderr.strip())
    if os.path.isdir(CODE_DIR):
        subprocess.run(['rm','-rf',CODE_DIR], check=True)
    if attempt == 5:
        raise RuntimeError('git clone failed 5x')
    time.sleep(10*attempt)

subprocess.run(['git','-C',CODE_DIR,'checkout',PINNED_COMMIT], check=True)
head = subprocess.run(['git','-C',CODE_DIR,'rev-parse','HEAD'],
                      capture_output=True, text=True, check=True).stdout.strip()
assert head.startswith(PINNED_COMMIT), f'{head} != {PINNED_COMMIT}'
print('commit:', head)

Clone OK (attempt 1)
commit: 3dfdb67edc1ae7bcbe7e5a3a7975f82ef11819ff


In [4]:
%cd {CODE_DIR}
!pip install -q hyperopt pandas tqdm scikit_learn pyyaml colorlog colorama tensorboard

/content/MBHT-KDD22


## 3. Đọc tiêu chí đã khoá + ghi bất đối xứng vào log TRƯỚC khi chạy

In [5]:
import json

with open(os.path.join(CODE_DIR, 'experiments', 'seeds.json')) as f:
    LOCK = json.load(f)

SEEDS   = LOCK['seeds']
PRIMARY = LOCK['primary_metric']
ALPHA   = LOCK['reporting']['alpha']

lock_commit = subprocess.run(
    ['git','-C',CODE_DIR,'log','-1','--format=%h %ci','--','experiments/seeds.json'],
    capture_output=True, text=True, check=True).stdout.strip()

print('seeds   :', SEEDS)
print('primary :', PRIMARY)
print('test    :', LOCK['reporting']['test'])
print('alpha   :', ALPHA)
print('seeds.json committed at:', lock_commit)
assert LOCK['locked_before_any_multiseed_run'] is True
assert LOCK['noise_floor'].get('tmall_beh') is None, \
    'San nhieu Tmall da co san -- pha nay phai tu do lai tu chinh no'

ASYMMETRY = {
    'recorded_before_any_result': True,
    'applies_only_if': 'compute forces an epoch cap; not applicable under full early-stopping',
    'negative_under_cap': 'WEAK evidence. B1 carries extra parameters and is handicapped by a '
                          'shortened budget, so a null result may be undertraining rather than '
                          'absence of effect. Must NOT be read as "B1 does not work".',
    'positive_under_cap': 'STRONG evidence. The effect appeared despite that handicap.',
}
path = os.path.join(RUN_META_DIR, 'reading_asymmetry_tmall.json')
with open(path, 'w') as f:
    json.dump(ASYMMETRY, f, indent=2)
print('\nDa khoa bat doi xung vao log truoc khi chay:', path)
for k, v in ASYMMETRY.items():
    print(f'  {k}: {v}')

seeds   : [2020, 2021, 2022]
primary : ndcg@10
test    : Welch's two-sample t-test (unequal variances, two-sided)
alpha   : 0.05
seeds.json committed at: 2755f88 2026-08-18 11:05:35 +0700

Da khoa bat doi xung vao log truoc khi chay: /content/drive/MyDrive/DeAnThS/run_meta/reading_asymmetry_tmall.json
  recorded_before_any_result: True
  applies_only_if: compute forces an epoch cap; not applicable under full early-stopping
  negative_under_cap: WEAK evidence. B1 carries extra parameters and is handicapped by a shortened budget, so a null result may be undertraining rather than absence of effect. Must NOT be read as "B1 does not work".
  positive_under_cap: STRONG evidence. The effect appeared despite that handicap.


## 4. Load Tmall — xác nhận |T| = 43 (suy ra, không hardcode)

In [6]:
from recbole.config import Config
from recbole.data import create_dataset
from recbole.data.utils import get_dataloader, create_samplers
from recbole.model.transition_utils import transition_vocab_size

# Config Tmall lay dung tu run_MBHT.py: scales [10,4,20] (retail moi la [5,4,20]).
# epochs = SAFETY_CAP, stopping_step = 10 -> early stopping quyet dinh, khong phai cap.
FROZEN = {
    'USER_ID_FIELD': 'session_id', 'load_col': None, 'neg_sampling': None,
    'benchmark_filename': ['train', 'test'], 'alias_of_item_id': ['item_id_list'],
    'topk': [5, 10, 101], 'metrics': ['Recall', 'NDCG', 'MRR'],
    'valid_metric': 'NDCG@10', 'eval_args': {'mode': 'full', 'order': 'TO'},
    'MAX_ITEM_LIST_LENGTH': 200,
    'train_batch_size': 64, 'eval_batch_size': 128,
    'hyper_len': 6, 'scales': [10, 4, 20],
    'enable_hg': 1, 'enable_ms': 1, 'customized_eval': 1, 'abaltion': '',
    'epochs': SAFETY_CAP, 'stopping_step': STOPPING_STEP,
}
base_cfg = {**FROZEN, 'data_path': DATASET_ROOT}

schema_config = Config(model='MBHT', dataset=DATASET_NAME, config_dict=base_cfg)
dataset = create_dataset(schema_config)

type_map = dataset.field2token_id['item_type_list']
n_types  = len(type_map)
T        = transition_vocab_size(n_types)
print('field2token_id[item_type_list] =', type_map)
print('n_types =', n_types, '  |T| =', T)
assert n_types == 6, f'Tmall phai co 6 type, dang thay {n_types}'
assert T == 43, f'|T| phai la 43, dang la {T}'
EXPECTED_PARAM_DELTA = T * 64
print('param delta ky vong =', EXPECTED_PARAM_DELTA, '(retail la 1984 -- khac nhau la dung)')


def fresh_split(cfg):
    """Mot ban du lieu sach cho MOT consumer. Khong dung chung: build() doi
    inter_feat tai cho (goi lan hai la crash), va train dataloader shuffle()
    split tai cho moi epoch nen run sau se khoi dau tu thu tu run truoc de lai."""
    ds = create_dataset(cfg)
    tr, te = ds.build()
    return ds, tr, te

/content/MBHT-KDD22/recbole/model/sequential_recommender/dien.py:306: SyntaxWarning: invalid escape sequence '\p'
  ..math: {h}_{t}^{\prime}=\left(1-a_{t}\right) * {h}_{t-1}^{\prime}+a_{t} * \tilde{{h}}_{t}^{\prime}
/content/MBHT-KDD22/recbole/model/sequential_recommender/dien.py:350: SyntaxWarning: invalid escape sequence '\p'
  ..math: \tilde{{u}}_{t}^{\prime}=a_{t} * {u}_{t}^{\prime} \\
/content/MBHT-KDD22/recbole/data/dataset/dataset.py:445: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[field].fillna(value='', inplace=T

field2token_id[item_type_list] = {np.str_('[PAD]'): 0, np.str_('2'): 1, np.str_('1'): 2, np.str_('0'): 3, np.str_('3'): 4, np.str_('4'): 5}
n_types = 6   |T| = 43
param delta ky vong = 2752 (retail la 1984 -- khac nhau la dung)


/content/MBHT-KDD22/recbole/data/dataset/sequential_dataset.py:146: FutureWarning: using <built-in function len> in Series.agg cannot aggregate and has been deprecated. Use Series.transform to keep behavior unchanged.
  self.inter_feat[self.item_list_length_field] = self.inter_feat[self.item_id_list_field].agg(len)


## 5. PASS #2 — param delta phải đúng 2,752

Làm trước run dài để nếu sai thì phát hiện ngay, không tốn giờ GPU.

In [7]:
import torch
from recbole.utils import init_seed
from recbole.model.sequential_recommender.mbht import MBHT

c_off = Config(model='MBHT', dataset=DATASET_NAME,
               config_dict={**base_cfg, 'seed': SEEDS[0], 'enable_transition_embedding': 0})
c_on  = Config(model='MBHT', dataset=DATASET_NAME,
               config_dict={**base_cfg, 'seed': SEEDS[0], 'enable_transition_embedding': 1})

pd_, ptr, pte = fresh_split(c_off)
ps, _ = create_samplers(c_off, pd_, [ptr, pte])
probe_loader = get_dataloader(c_off, 'train')(c_off, ptr, ps, shuffle=False)
BATCHES_PER_EPOCH = len(probe_loader)
print('batches/epoch (Tmall):', BATCHES_PER_EPOCH,
      f'  -> {RETAIL_STEPS/BATCHES_PER_EPOCH:.2f} epoch Tmall = ca qua trinh RetailRocket')

init_seed(SEEDS[0], True); m_off = MBHT(c_off, probe_loader.dataset)
init_seed(SEEDS[0], True); m_on  = MBHT(c_on,  probe_loader.dataset)
d = sum(p.numel() for p in m_on.parameters()) - sum(p.numel() for p in m_off.parameters())
print(f'\nparam delta = {d}   (ky vong {EXPECTED_PARAM_DELTA} = |T| 43 x 64)')
assert d == EXPECTED_PARAM_DELTA, f'PASS #2 FAIL: {d} != {EXPECTED_PARAM_DELTA}'

sd_off, sd_on = m_off.state_dict(), m_on.state_dict()
assert set(sd_on) - set(sd_off) == {'transition_embedding.weight'}
bad = [k for k in sd_off if not torch.equal(sd_off[k], sd_on[k])]
assert not bad, f'bat B1 lam lech tham so A0: {bad}'
print('PASS #2 OK')
del m_off, m_on, probe_loader, pd_, ptr, pte

/content/MBHT-KDD22/recbole/data/dataset/dataset.py:445: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[field].fillna(value='', inplace=True)
/content/MBHT-KDD22/recbole/data/dataset/dataset.py:445: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].me

batches/epoch (Tmall): 7234   -> 1.73 epoch Tmall = ca qua trinh RetailRocket

param delta = 2752   (ky vong 2752 = |T| 43 x 64)
PASS #2 OK


## 6. Hàm chạy một run — early-stopping bật, in đường validation từng epoch

`trainer.fit()` nhận `callback_fn(epoch_idx, valid_score)`; ta dùng nó để lấy `ndcg@10` sau
mỗi epoch, kèm số bước gradient tích luỹ và thời gian thực — đủ để nhìn ra điểm hội tụ **ngay
trong lúc chạy**, không phải đợi hết run.

In [8]:
import glob, hashlib, time as _time
from logging import getLogger
from recbole.utils import init_logger, get_trainer, set_color

def run(arm, seed, verbose_curve=True):
    """arm: 'A0' | 'A1'. Bo qua neu da co ket qua trong run_meta."""
    enable_b1 = (arm == 'A1')
    tag  = f'{arm}-{DATASET_NAME}-seed{seed}'
    meta = os.path.join(RUN_META_DIR, f'{tag}.json')
    if os.path.exists(meta):
        with open(meta) as f:
            done = json.load(f)
        print(f'[{tag}] da co ket qua, bo qua  (epoch chay: {done.get("epochs_ran")})')
        return done

    ckpt = os.path.join(CKPT_DIR, tag); os.makedirs(ckpt, exist_ok=True)
    logd = os.path.join(LOG_DIR, tag);  os.makedirs(logd, exist_ok=True)

    cfg = Config(model='MBHT', dataset=DATASET_NAME, config_dict={
        **base_cfg, 'checkpoint_dir': ckpt, 'gpu_id': 0, 'seed': seed,
        'enable_transition_embedding': int(enable_b1)})
    init_seed(cfg['seed'], cfg['reproducibility'])
    init_logger(cfg, log_root=logd)
    getLogger().info(f'TAG={tag} commit={PINNED_COMMIT} arm={arm} seed={seed} '
                     f'patience={STOPPING_STEP} safety_cap={SAFETY_CAP}')

    ds, tr, te = fresh_split(cfg)
    s_tr, s_te = create_samplers(cfg, ds, [tr, te])
    train_data = get_dataloader(cfg, 'train')(cfg, tr, s_tr, shuffle=True)
    test_data  = get_dataloader(cfg, 'test')(cfg, te, s_te, shuffle=False)
    n_batches = len(train_data)

    model = MBHT(cfg, train_data.dataset).to(cfg['device'])
    n_params = sum(p.numel() for p in model.parameters())
    print(f'[{tag}] |T|={model.n_transitions} n_types={model.n_behavior_types} '
          f'params={n_params} batches/epoch={n_batches}')

    curve, t0 = [], _time.time()
    if verbose_curve:
        print(f'\n{"epoch":>6} {"ndcg@10":>9} {"delta":>9} {"buoc grad":>11} '
              f'{"vs retail":>10} {"phut":>7} {"du kien 15ep":>13}')

    def on_epoch(epoch_idx, valid_score):
        elapsed = _time.time() - t0
        steps = (epoch_idx + 1) * n_batches
        prev = curve[-1][1] if curve else None
        curve.append((epoch_idx, float(valid_score), steps, elapsed))
        if verbose_curve:
            dtxt = f'{valid_score - prev:+.4f}' if prev is not None else '    --'
            per_ep = elapsed / (epoch_idx + 1)
            print(f'{epoch_idx:>6} {valid_score:>9.4f} {dtxt:>9} {steps:>11,} '
                  f'{steps/RETAIL_STEPS:>9.2f}x {elapsed/60:>7.1f} '
                  f'{per_ep*SAFETY_CAP/3600:>12.1f}h')

    trainer = get_trainer(cfg['MODEL_TYPE'], cfg['model'])(cfg, model)
    prev_ck = sorted(glob.glob(os.path.join(ckpt, '*.pth')), key=os.path.getmtime)
    if prev_ck:
        print(f'[{tag}] resuming from {prev_ck[-1]}')
        trainer.resume_checkpoint(prev_ck[-1])

    _, result = trainer.fit(train_data, test_data, saved=True,
                            show_progress=cfg['show_progress'], callback_fn=on_epoch)
    if result is None:
        raise RuntimeError(f'[{tag}] khong co ket qua: resume tu checkpoint da train xong. '
                           f'Xoa {ckpt} roi chay lai.')

    epochs_ran = len(curve)
    hit_cap = epochs_ran >= SAFETY_CAP
    h = hashlib.sha256()
    for k, v in sorted(model.state_dict().items()):
        h.update(k.encode()); h.update(v.detach().cpu().numpy().tobytes())

    out = {'tag': tag, 'arm': arm, 'seed': seed, 'commit': PINNED_COMMIT,
           'patience': STOPPING_STEP, 'safety_cap': SAFETY_CAP,
           'epochs_ran': epochs_ran, 'hit_safety_cap': hit_cap,
           'batches_per_epoch': n_batches, 'grad_steps': epochs_ran * n_batches,
           'grad_steps_vs_retail': epochs_ran * n_batches / RETAIL_STEPS,
           'wall_hours': (_time.time() - t0) / 3600.0,
           'n_params': n_params, 'param_hash': h.hexdigest()[:16],
           'valid_curve': [{'epoch': e, 'ndcg@10': s, 'grad_steps': st} for e, s, st, _ in curve],
           'result': {k: float(v) for k, v in result.items()}}
    with open(meta, 'w') as f:
        json.dump(out, f, indent=2)
    print(set_color(f'\n[{tag}] ', 'yellow') + json.dumps(out['result']))
    print(f'[{tag}] epoch chay={epochs_ran}  cham safety cap={hit_cap}  '
          f'{out["wall_hours"]:.1f}h')
    return out

## 7. PROBE = A0 seed 2020 — vừa là probe hội tụ, vừa là run pha 1 thật

Không phí: đây là một trong 3 run A0 của pha 1.

**Vừa chạy vừa theo dõi bảng.** Nếu `ndcg@10` phẳng nhiều epoch liên tiếp thì model đã hội tụ —
early-stopping cần 10 epoch không cải thiện mới kích hoạt, nên nó còn chạy thêm một lúc sau khi
đường đã phẳng. Cột `du kien 15ep` cho biết tổng chi phí nếu chạy hết safety cap.

In [ ]:
probe = run('A0', SEEDS[0])

curve = probe['valid_curve']
best_ep = max(range(len(curve)), key=lambda i: curve[i]['ndcg@10'])
print('\n' + '='*68)
print('KET QUA PROBE')
print('='*68)
print(f'  epoch da chay        : {probe["epochs_ran"]}')
print(f'  cham safety cap ({SAFETY_CAP})? : {probe["hit_safety_cap"]}')
print(f'  epoch tot nhat       : {best_ep}  (ndcg@10 = {curve[best_ep]["ndcg@10"]:.4f})')
print(f'  buoc gradient        : {probe["grad_steps"]:,}  '
      f'= {probe["grad_steps_vs_retail"]:.2f}x ca qua trinh RetailRocket')
print(f'  thoi gian            : {probe["wall_hours"]:.1f} gio')
print()
if probe['hit_safety_cap']:
    print('  -> CHAM SAFETY CAP. Early-stopping CHUA tu kich hoat.')
    print('     DUNG LAI. Gui bang duong validation ve de ban danh doi bang du lieu that.')
else:
    print('  -> Early-stopping TU KICH HOAT truoc cap.')
    print('     Day la protocol GOC day du, so duoc voi so published.')
    print('     Gui ket qua ve de duyet chay 5 run con lai.')
print('='*68)

/content/MBHT-KDD22/recbole/data/dataset/dataset.py:445: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[field].fillna(value='', inplace=True)
/content/MBHT-KDD22/recbole/data/dataset/dataset.py:445: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].me

[A0-tmall_beh-seed2020] |T|=43 n_types=6 params=6619128 batches/epoch=7234

 epoch   ndcg@10     delta   buoc grad  vs retail    phut  du kien 15ep


Evaluate   : 100%|█████████████████████████| 79/79 [01:34<00:00,  1.19s/it, GPU RAM: 4.01 G/14.56 G]


     0    0.1280        --       7,234      0.58x   121.2         30.3h


Evaluate   : 100%|█████████████████████████| 79/79 [01:33<00:00,  1.19s/it, GPU RAM: 4.01 G/14.56 G]


     1    0.2394   +0.1114      14,468      1.16x   240.5         30.1h


Train     2:  22%|█████████▋                                  | 1600/7234 [26:30<1:44:41,  1.11s/it]

## 8. ⛔ DỪNG Ở ĐÂY — gửi kết quả probe về trước khi chạy tiếp

Cần gửi:
1. Bảng đường validation từng epoch (cell 7).
2. Khối `KET QUA PROBE`.
3. `PASS #2` param delta = 2,752 (cell 5).

**Chưa chạy cell dưới.** 5 run còn lại tốn nhiều giờ GPU; chỉ chạy sau khi đã xác nhận hội tụ.

## 9. (SAU KHI DUYỆT) 5 run còn lại

In [ ]:
results = {'A0': [probe], 'A1': []}
for seed in SEEDS[1:]:
    results['A0'].append(run('A0', seed)); print()
for seed in SEEDS:
    results['A1'].append(run('A1', seed)); print()
print('Xong:', {a: len(v) for a, v in results.items()})

## 10. Phân tích — mean±std, sàn nhiễu Tmall, Welch's t-test

In [ ]:
import statistics
from scipy import stats

vals = {arm: [r['result'][PRIMARY] for r in results[arm]] for arm in ('A0','A1')}
noise_tmall = max(vals['A0']) - min(vals['A0'])   # do tu chinh Tmall, khong dung so retail

print(f'=== {PRIMARY} tren {DATASET_NAME}, seeds={SEEDS} ===\n')
for arm in ('A0','A1'):
    v = vals[arm]
    print(f'{arm}: ' + '  '.join(f'{x:.4f}' for x in v) +
          f'   mean={statistics.mean(v):.4f}  std={statistics.stdev(v):.4f}')
print('\nepoch chay:', {a: [r['epochs_ran'] for r in results[a]] for a in ('A0','A1')})
print('cham safety cap:', {a: [r['hit_safety_cap'] for r in results[a]] for a in ('A0','A1')})

diff = statistics.mean(vals['A1']) - statistics.mean(vals['A0'])
t, p = stats.ttest_ind(vals['A1'], vals['A0'], equal_var=False)
print(f'\nsan nhieu Tmall (range 3 run A0) : {noise_tmall:.4f}')
print(f'hieu A1 - A0                     : {diff:+.4f}')
print(f"Welch's t-test                   : t={t:+.3f}  p={p:.4f}  (alpha={ALPHA})")

c1, c2 = abs(diff) > noise_tmall, p < ALPHA
capped = any(r['hit_safety_cap'] for r in results['A0'] + results['A1'])
print(f'\n  (1) hieu vuot san nhieu ? {c1}')
print(f'  (2) p < {ALPHA} ?           {c2}')
print('\n' + '='*68)
if c1 and c2:
    print(f'GO -- B1 {"tang" if diff>0 else "GIAM"} {PRIMARY} vuot ca nhieu lan y nghia.')
    if diff < 0: print('Luu y: huong AM -- B1 lam giam metric mot cach co y nghia.')
    if capped: print('Co run cham safety cap: theo bat doi xung da khoa, DUONG tinh la bang chung MANH.')
else:
    print('TRONG NHIEU / KHONG CO Y NGHIA -- KET QUA THAT, khong phai loi.')
    if capped:
        print('CANH BAO: co run cham safety cap -> chua hoi tu.')
        print('Theo bat doi xung da khoa TRUOC khi chay: AM tinh duoi cap la bang chung YEU.')
        print('KHONG duoc ket luan "B1 vo dung" tu day.')
    else:
        print('Moi run deu early-stop tu nhien -> da hoi tu, ket luan nay vung.')
    print('Khong chay them seed de "tim" ket qua dep hon. Muon mo rong phai khai bao')
    print('seeds_phase2 trong experiments/seeds.json TRUOC, commit, roi bao ca hai pha.')
print('='*68)

summary = {'dataset': DATASET_NAME, 'seeds': SEEDS, 'primary_metric': PRIMARY,
           'patience': STOPPING_STEP, 'safety_cap': SAFETY_CAP,
           'any_run_hit_cap': capped, 'values': vals,
           'mean': {a: statistics.mean(v) for a, v in vals.items()},
           'std': {a: statistics.stdev(v) for a, v in vals.items()},
           'epochs_ran': {a: [r['epochs_ran'] for r in results[a]] for a in ('A0','A1')},
           'noise_floor_tmall': noise_tmall, 'diff': diff,
           'welch_t': float(t), 'welch_p': float(p), 'alpha': ALPHA,
           'exceeds_noise': bool(c1), 'significant': bool(c2),
           'verdict': 'GO' if (c1 and c2) else 'WITHIN_NOISE_OR_NS'}
with open(os.path.join(RUN_META_DIR, f'phase1_summary_{DATASET_NAME}.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('\nDa luu summary.')

## 11. Bảng đầy đủ mọi metric (tham khảo)

In [ ]:
all_keys = sorted(results['A0'][0]['result'])
print(f'{"metric":>12} {"A0 mean":>10} {"A0 std":>9} {"A1 mean":>10} {"A1 std":>9} {"diff":>9} {"p":>8}')
for k in all_keys:
    a0 = [r['result'][k] for r in results['A0']]
    a1 = [r['result'][k] for r in results['A1']]
    pk = float('nan') if (statistics.stdev(a0)==0 and statistics.stdev(a1)==0) \
         else stats.ttest_ind(a1, a0, equal_var=False).pvalue
    star = ' *' if k == PRIMARY else ''
    print(f'{k:>12} {statistics.mean(a0):>10.4f} {statistics.stdev(a0):>9.4f} '
          f'{statistics.mean(a1):>10.4f} {statistics.stdev(a1):>9.4f} '
          f'{statistics.mean(a1)-statistics.mean(a0):>+9.4f} {pk:>8.4f}{star}')
print('\n* = metric chinh da khoa truoc. Cac dong khac chi tham khao;')
print('  doc nhieu metric roi chon cai dep nhat la multiple-comparison, reviewer se bat.')